# Cross-branch DPO-delta transfer — Stage 2

**This is the actual research question.** Stage 1 only established a prerequisite: that injecting a branch's OWN measured DPO delta at layer 24 moves its behaviour toward its own post-DPO profile (it passed, at all three coefficients, in quadrant C).

Stage 2 asks: **does the DPO delta measured in the Alpaca branch transfer to the Dolly branch?** I.e. is the DPO-induced activation change reusable across two different upstream training paths, or is it path-specific?

**6 core units, all at coefficient 1.0, 414 prompts each:**

| Condition | What it tests |
|---|---|
| `xfer_delta_source_identity` | **The primary arm.** B2 + Δ_A (Alpaca's delta, identity transfer) |
| `xfer_delta_source_shuf_wq` | Within-quadrant shuffled Δ_A — destroys the prompt↔delta pairing, keeps the distribution |
| `xfer_delta_source_normmatched` | Per-row random matched to ‖Δ_A‖ — is it the delta, or just a perturbation of that size? |
| `xfer_delta_source_dosematched` | Δ_A rescaled per row to ‖Δ_B‖ — separates transferability from magnitude. **Stronger oracle:** uses the target branch's own delta norm. |
| `dir_source_matched` | s_A·d_A — Alpaca's generic A–D refusal *direction*, same injector/site/dose |
| `dir_target_matched` | s_B·d_B — Dolly's own direction, the within-branch concept reference |

The last two are what let us separate **"the safety concept transfers"** from **"the specific DPO change transfers"** — they go through the identical per-row injector at the identical position and a calibration-only dose, so they differ from the delta arms only in *which vector* is injected.

**Reuses everything from Stage 1**: same activations, same benchmark, same split, same hooks, same gate machinery. Stage-1 outputs are untouched.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = 'c784139b156910b717a73ae7ae5e1f2cca803c98'

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
print("checked out", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2b. HF authentication (required for the section-10 run)

The section-10 runner shells out `!python -m src.crossbranch.worker`
once per unit. Each worker calls `load_stage_model()`, which pulls the base
model + adapters from the HF Hub. **Unauthenticated** Hub requests are rate
limited: 6 cold model loads back-to-back hit HTTP 429 and every unit fails at
load time before generating anything (this is exactly what the first Stage-2
run hit -- all 6 units `returncode 1`, no `raw/` produced).

Setting `HF_TOKEN` in `os.environ` fixes it and is inherited by every
`!python -m ...` subprocess. Set the `HF_TOKEN` Colab secret first (key icon
in the left sidebar), then run this cell.

In [ ]:
import os

try:
    from google.colab import userdata
    from huggingface_hub import login
    _tok = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = _tok
    login(token=_tok)
    print('HF login OK')
except Exception as e:
    print('HF NOT authenticated:', repr(e))
    print('Set the HF_TOKEN Colab secret (key icon, left sidebar) and rerun '
          'this cell. Without it the 6-unit run in section 10 will rate-limit '
          'and fail at model-load time.')

## 3. Apply the crossbranch patch

Upload `crossbranch_p0_patch.zip` (from your Downloads — rebuilt to include the Stage-2 builders and the sensitivity module).

In [ ]:
from google.colab import files
import zipfile, io

uploaded = files.upload()
assert len(uploaded) == 1, "upload exactly one file: crossbranch_p0_patch.zip"
name, data = next(iter(uploaded.items()))
with zipfile.ZipFile(io.BytesIO(data)) as z:
    names = z.namelist()
    assert all(n.startswith(('src/crossbranch/', 'tests/crossbranch/')) for n in names), (
        "patch contains files outside the crossbranch package — refusing to apply"
    )
    z.extractall('.')
print(f"applied {len(names)} files from {name}")

## 4. Dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip uninstall -y torchao || true
!nvidia-smi

## 5. Copy activations AND the two direction vectors from Drive

Stage 2 needs everything Stage 1 needed, **plus** the A–D direction vectors for M3 and M3_alt (for the two concept arms). Set the same path you used in the Stage-1 notebook.

In [ ]:
RESULTS_SOURCE_DIR = '/content/drive/MyDrive/dpo_v2/results'

import shutil
from pathlib import Path

src_root = Path(RESULTS_SOURCE_DIR)
assert (src_root / 'activations').exists(), f"{src_root}/activations not found — fix RESULTS_SOURCE_DIR"

STAGES = ('M2', 'M3', 'M2_alt', 'M3_alt')
ACT_SUFFIXES = ('_final.npy', '_pooled.npy', '_metadata.json', '_metadata_binding.json')
DIR_STAGES = ('M3', 'M3_alt')
DIR_SUFFIXES = ('_direction_654.npy', '_direction_654_binding.json')

copied, missing = [], []

dst = Path('results/activations'); dst.mkdir(parents=True, exist_ok=True)
for stage in STAGES:
    for suf in ACT_SUFFIXES:
        s = src_root / 'activations' / f'{stage}{suf}'
        (copied if s.exists() else missing).append(s.name)
        if s.exists():
            shutil.copy2(s, dst / f'{stage}{suf}')

dst_dir = Path('results/refusal_direction'); dst_dir.mkdir(parents=True, exist_ok=True)
for stage in DIR_STAGES:
    for suf in DIR_SUFFIXES:
        s = src_root / 'refusal_direction' / f'{stage}{suf}'
        (copied if s.exists() else missing).append(s.name)
        if s.exists():
            shutil.copy2(s, dst_dir / f'{stage}{suf}')

print(f"copied {len(copied)} files")
if missing:
    print(f"MISSING ({len(missing)}):")
    for m in missing:
        print("  ", m)

## 6. Preflight — activations and directions both bound to the frozen benchmark

In [ ]:
import json
from pathlib import Path
import numpy as np
from src.v2_io import load_run_inputs, identity_snapshot, load_json

bp, bsha, sp, ssha = load_run_inputs(None, None, 'logs/direction_split_manifest.json')
rows = [json.loads(l) for l in Path(bp).read_text(encoding='utf-8').splitlines() if l.strip()]
snap = identity_snapshot(rows)
print(f"benchmark {bsha[:12]}...  split {ssha[:12]}...  rows {len(rows)}\n")

ok = True
for stage in ('M2', 'M3', 'M2_alt', 'M3_alt'):
    arr = np.load(f'results/activations/{stage}_final.npy', mmap_mode='r')
    meta = load_json(f'results/activations/{stage}_metadata.json')
    bind = load_json(f'results/activations/{stage}_metadata_binding.json')
    good = (arr.shape[0] == len(rows) and meta == snap
            and bind.get('benchmark_sha256') == bsha
            and bind.get('split_manifest_sha256') == ssha)
    ok &= good
    print(f"  activation {stage:8s} {str(arr.shape):18s} {'PASS' if good else 'FAIL'}")

for stage in ('M3', 'M3_alt'):
    bind = load_json(f'results/refusal_direction/{stage}_direction_654_binding.json')
    d = np.load(f'results/refusal_direction/{stage}_direction_654.npy')
    norm = float(np.linalg.norm(d[24]))
    good = (bind.get('benchmark_sha256') == bsha
            and bind.get('split_manifest_sha256') == ssha
            and abs(norm - 1.0) < 1e-3)
    ok &= good
    print(f"  direction  {stage:8s} layer24 norm={norm:.6f}  {'PASS' if good else 'FAIL'}")

assert ok, "preflight FAILED — do not proceed"
print("\nAll PASS.")

## 7. Test gate

In [ ]:
!python -m pytest tests/crossbranch -q

## 8. Assemble P0 + Stage-2 vectors (CPU, seconds)

Rebuilds the four P0 vectors (bit-identical to Stage 1 — the seed stream is append-only, verified locally) and adds the five Stage-2 ones. Prints the calibration-only direction doses.

In [ ]:
!python -m src.crossbranch.delta --stage2

## 9. Dry-run — confirm the 6 core units are ready

In [ ]:
STAGE2_CORE = (
    "xfer_delta_source_identity xfer_delta_source_shuf_wq "
    "xfer_delta_source_normmatched xfer_delta_source_dosematched "
    "dir_source_matched dir_target_matched"
)
!python -m src.crossbranch.runner --dry-run --allow-stage2 \
    --conditions {STAGE2_CORE} --coefficients 1.0

## 10. Stage 2 — the run (6 units × 414 prompts)

`--allow-stage2` is required and deliberate: the worker refuses Stage-2 conditions without it, so no stray flag can start this by accident.

In [ ]:
!python -m src.crossbranch.runner --allow-stage2 \
    --conditions {STAGE2_CORE} --coefficients 1.0

## 11. Package everything (Stage 1 + Stage 2) to bring back

In [ ]:
import shutil
shutil.make_archive('/content/crossbranch_stage2_results', 'zip', 'results/crossbranch')

from google.colab import files
files.download('/content/crossbranch_stage2_results.zip')

---
## STOP — analysis happens back on the local machine

Bring `crossbranch_stage2_results.zip` back. The Stage-2 interpretation is deliberately **not** run here: unlike Stage 1 there is no single pass/fail gate, it is a comparison across six arms that needs to be read carefully, alongside the same kind of manual reading of raw completions we did for Stage 1.

The question it answers, and the shape of the possible answers:

- **`xfer_delta_source_identity` moves toward B3, and shuffled/random do not** → evidence that part of the DPO change is reusable across upstream training paths.
- **It does not move, but `dir_source_matched` does** → the safety *concept* transfers while the specific DPO *change* does not — path-specific coupling.
- **Shuffled or random move as much as identity** → magnitude/perturbation effect, not transfer.
- **Nothing moves** → the delta is path-specific at this site; report plainly, and note the protocol bound (L24, final prompt position, additive).